In [1]:
# system imports
import os
import sys

sys.path.append(os.path.abspath(".."))

# Import modules
import networkx as nx
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Import your modular code
from src.graph import Graph
from src.a_star import a_star


In [4]:

# --- 3. Project-Specific Imports ---
# NOTE: The ImportError handling is kept to alert the user if files are missing.
try:
    # We now import directly because './src' is on the path (or should be).
    from src.graph import Graph  
    from src.a_star import a_star
except ImportError as e:
    print(f"Error loading project modules: {e}")
    print("Please ensure src/graph.py and src/a_star.py exist and are correct.")
    # Define dummy functions to prevent total crash if modules are missing
    class Graph:
        def __init__(self, *args): self.adj = {}
    def a_star(*args): return None, float('inf')

# --- 4. Load the Graph ---
# Use the correct path relative to the notebook. 
# Since the modules are loaded, the path to the JSON should be updated.
try:
    # Assuming standard structure: data folder is a sibling to src folder
    graph = Graph("../data/stations.json")
    stations = list(graph.adj.keys())
except FileNotFoundError:
    print("Error: data/stations.json not found. Please check the file path.")
    stations = []


# --- 5. Interactive Widgets for Start and Goal Selection ---
start_dropdown = widgets.Dropdown(
    options=stations,
    description='Start:',
    style={'description_width': 'initial'}
)

goal_dropdown = widgets.Dropdown(
    options=stations,
    description='Goal:',
    style={'description_width': 'initial'}
)

button = widgets.Button(description="Find Optimal Path 🗺️")

output = widgets.Output()


In [5]:
# --- 6. Visualization Function (The Core Logic) ---
def visualize_path(b):
    """Handles the button click, runs A*, and displays the results."""
    
    # Clear previous output
    output.clear_output() 
    start = start_dropdown.value
    goal = goal_dropdown.value
    
    # Check for valid stations before running the algorithm
    if not start or not goal:
        with output:
             print("Please select both a Start and a Goal station.")
        return

    # Run A* algorithm
    path, cost = a_star(graph, start, goal)
    
    with output:
        if path:
            print(f"✅ Optimal Path: {' -> '.join(path)}")
            print(f"Cost: {cost} units (Distance/Time)")
            
            # Create NetworkX graph (NEW ROBUST CREATION)
            G = nx.Graph()
            # Iterate through the fully connected adjacency list and add edges with 'weight'
            # We use a set to track edges already added to prevent duplicates (since graph.adj is bidirectional)
            added_edges = set() 
            for u, neighbors in graph.adj.items():
                for v, cost in neighbors.items():
                    # Check if the reverse edge has already been added
                    if (v, u) not in added_edges:
                        G.add_edge(u, v, weight=cost)
                        added_edges.add((u, v))
            
            # Layout for visualization (keep seed for consistent layout)
            pos = nx.spring_layout(G, seed=42)
            
            # Matplotlib figure setup
            plt.figure(figsize=(12, 8))
            
            # 1. Draw all nodes and edges (base map)
            nx.draw_networkx_nodes(G, pos, node_color="skyblue", node_size=2000)
            nx.draw_networkx_labels(G, pos, font_size=10)
            nx.draw_networkx_edges(G, pos, edge_color="lightgray")
            
            # 2. Add edge weights (NOW we can use the canonical 'weight' attribute)
            edge_labels = nx.get_edge_attributes(G, 'weight')
            nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='black', font_size=9)
            
            # 3. Highlight the path
            edge_path = list(zip(path, path[1:]))
            nx.draw_networkx_edges(G, pos, edgelist=edge_path, edge_color="orange", width=4, style='dashed')
            nx.draw_networkx_nodes(G, pos, nodelist=path, node_color="gold", node_size=2500)
            
            # 4. Highlight Start/Goal points specifically
            nx.draw_networkx_nodes(G, pos, nodelist=[start], node_color="green", node_size=3000, label="Start", linewidths=2, edgecolors='black')
            nx.draw_networkx_nodes(G, pos, nodelist=[goal], node_color="red", node_size=3000, label="Goal", linewidths=2, edgecolors='black')
            
            plt.title(f"Optimal Taxi Route: {start} to {goal}", fontsize=14)
            plt.axis('off') # Hide axes
            plt.show() # Display the plot
        else:
            print(f"❌ No path found from {start} to {goal}!")

# --- 7. Link the Button to the Function ---
button.on_click(visualize_path)

# --- 8. Display the User Interface ---
# Use HBox/VBox to arrange the widgets nicely
ui = widgets.VBox([
    widgets.HTML("<h3>Addis Ababa Taxi Route Finder</h3>"),
    widgets.HBox([start_dropdown, goal_dropdown, button]),
    output
])

display(ui)